# NLP-Reco: End-to-End Demo

This notebook walks through the full hotel recommendation pipeline step by step.

## Problem Statement

Given a set of Turkish hotel descriptions and ratings, recommend the **top 10 most similar hotels** for each hotel in the catalogue.

## Two-Stage Architecture

```
Hotel descriptions (Turkish text)
          │
          ▼
  ┌───────────────┐
  │ Text Pipeline │  clean → normalize → tokenize → remove stopwords → lemmatize
  └───────┬───────┘
          │
          ▼
  ┌───────────────┐
  │   BERTopic    │  multilingual topic modeling → topic embeddings per hotel
  └───────┬───────┘
          │
          ▼
  ┌───────────────────────────────────┐
  │  Stage 1 — Retrieval             │  cosine similarity → top-10 candidates
  ├───────────────────────────────────┤
  │  Stage 2 — Ranking               │  re-rank by popularity score
  └───────────────────────────────────┘
          │
          ▼
  hotel_recommendations.csv
```

## Design Decisions

| Decision | Choice | Rationale |
|---|---|---|
| Topic model | BERTopic (multilingual) | Better semantic coherence than LDA on short texts |
| Lemmatizer | Zemberek (or Fallback) | Turkish agglutination requires morphological analysis |
| Popularity signal | comments + images (weighted) | Proxy for social proof; decoupled from ratings |
| Missing embeddings | Fallback to popularity ranking | Graceful degradation instead of crashing |

## 1. Setup

Set `PROCESSOR_MODE` based on your environment:
- `"fallback"` — works anywhere, no Java needed  
- `"turkish"` — full Zemberek morphological analysis (requires Java + `jars/zemberek-full.jar`)

In [ ]:
import sys
import os

# Make sure project root is on the path when running from notebooks/
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

PROCESSOR_MODE = "fallback"  # change to "turkish" if Zemberek is installed

print(f"Using processor: {PROCESSOR_MODE}")

## 2. Data Loading & Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
import src.config as config
from src.data_preprocessing import load_data, clean_hotel_info, clean_session_data, compute_weighted_star_rating, compute_popularity_score

hotel_info_raw, hotel_desc, session_data = load_data()
hotel_info = clean_hotel_info(hotel_info_raw.copy())
session_data = clean_session_data(session_data)

print(f"Hotels: {len(hotel_info)}")
print(f"Descriptions: {len(hotel_desc)}")
print(f"Sessions: {len(session_data)}")
hotel_info.head(3)

In [ ]:
# Weighted star rating formula:
#   weighted = 0.30*general + 0.25*food + 0.10*cleaning + 0.10*location
#            + 0.10*service + 0.05*wifi + 0.05*condition + 0.05*price
# Then cap at 99th percentile and normalize to [1, 5]

hotel_info = compute_weighted_star_rating(hotel_info)
hotel_info = compute_popularity_score(hotel_info)

print("Star rating range:", hotel_info['normalized_star_rating'].min().round(2), "–", hotel_info['normalized_star_rating'].max().round(2))
print("Popularity score range:", hotel_info['popularity_score'].min().round(3), "–", hotel_info['popularity_score'].max().round(3))
hotel_info[['hotel_name', 'hotel_city', 'normalized_star_rating', 'popularity_score']].sort_values('popularity_score', ascending=False).head(5)

## 3. Text Processing

We demonstrate each stage of the pipeline on a single Turkish sentence.

In [ ]:
from utils.text_processing import get_text_processor

processor = get_text_processor(PROCESSOR_MODE)

sample = "Odalar geniş ve temizdi.<br/>Personel çok ilgiliydi. Denize 2 km mesafede."
print("Original:        ", sample)
print("clean_text():    ", processor.clean_text(sample))

cleaned = processor.clean_text(sample)
print("normalize():     ", processor.normalize(cleaned))

normalized = processor.normalize(cleaned)
tokens = processor.tokenize(normalized)
print("tokenize():      ", tokens)

no_stop = processor.remove_stopwords(tokens)
print("remove_stopwords:", no_stop)

lemmas = processor.lemmatize(no_stop)
print("lemmatize():     ", lemmas)

print("process_text():  ", processor.process_text(sample))

In [ ]:
from utils.text_processing import process_hotel_descriptions

hotel_desc_processed = process_hotel_descriptions(hotel_desc.copy(), processor=processor)
hotel_desc_processed[['hotel_id', 'description_text', 'cleaned_description', 'lemmatized_text']].head(3)

## 4. Topic Modeling

In [ ]:
from src.topic_modeling import merge_hotel_descriptions, train_topic_model, extract_topic_embeddings

hotel_desc_merged = merge_hotel_descriptions(hotel_desc_processed)
print(f"Merged descriptions: {len(hotel_desc_merged)} hotels")
hotel_desc_merged.head(3)

In [ ]:
hotel_desc_merged, topic_model = train_topic_model(hotel_desc_merged)
hotel_desc_merged = extract_topic_embeddings(hotel_desc_merged, topic_model)

# Show discovered topics
topic_info = topic_model.get_topic_info()
print(f"Topics discovered: {len(topic_info) - 1} (excluding outlier topic -1)")
topic_info.head(10)

In [ ]:
# Topic distribution across hotels
import matplotlib.pyplot as plt

topic_counts = hotel_desc_merged['topic'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8, 3))
topic_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Topic ID')
ax.set_ylabel('Number of Hotels')
ax.set_title('Topic Distribution Across Hotels')
plt.tight_layout()
plt.show()

## 5. Recommendations

In [ ]:
from src.recommendation import (
    clean_and_convert_embedding, compute_cosine_similarity,
    generate_recommendations, handle_missing_embeddings, rank_recommendations
)

final_df = hotel_desc_merged.merge(
    hotel_info[['hotel_id', 'normalized_star_rating', 'popularity_score']],
    on='hotel_id', how='left'
)
final_df['topic_embedding'] = final_df['topic_embedding'].apply(clean_and_convert_embedding)

cosine_sim_matrix, valid_indices = compute_cosine_similarity(final_df)
recommendations = generate_recommendations(final_df, cosine_sim_matrix, valid_indices)
recommendations = handle_missing_embeddings(final_df, recommendations, valid_indices)
recommendation_df = rank_recommendations(final_df, recommendations)

print(f"Recommendation rows: {len(recommendation_df)}")
recommendation_df.head(5)

In [ ]:
# Show recommendations for 3 sample hotels in a readable table
sample_hotels = hotel_info['hotel_id'].sample(3, random_state=42).tolist()

for source_id in sample_hotels:
    source_name = hotel_info.loc[hotel_info['hotel_id'] == source_id, 'hotel_name'].values[0]
    source_city = hotel_info.loc[hotel_info['hotel_id'] == source_id, 'hotel_city'].values[0]
    print(f"\n{'='*60}")
    print(f"Source: {source_name} ({source_city})")
    print(f"{'='*60}")

    recos = recommendation_df[recommendation_df['hotel_id'] == source_id].merge(
        hotel_info[['hotel_id', 'hotel_name', 'hotel_city', 'normalized_star_rating', 'popularity_score']],
        left_on='reco_hotel_id', right_on='hotel_id'
    )[['rank', 'hotel_name', 'hotel_city', 'normalized_star_rating', 'popularity_score']]

    print(recos.to_string(index=False))

## 6. Validation — City Divergence

In [ ]:
from src.validation import merge_hotel_metadata, compute_city_divergence

# Save first so validate_recommendations() can load it
recommendation_df.to_csv(f"{config.DATA_PATH}/hotel_recommendations.csv", index=False)

merged_df = merge_hotel_metadata(recommendation_df, hotel_info)
city_divergence, overall_divergence_ratio = compute_city_divergence(merged_df)

print(f"Overall divergence ratio: {overall_divergence_ratio:.2%}")
print(f"→ {1 - overall_divergence_ratio:.0%} of recommendations are from the same city as the source hotel.")
city_divergence.sort_values('divergence_ratio', ascending=False).head(10)

In [ ]:
# Self-recommendation check: no hotel should recommend itself
self_recos = merged_df.query("hotel_id == reco_hotel_id")
assert self_recos.empty, f"Found {len(self_recos)} self-recommendations!"
print("Self-recommendation check passed.")

## 7. Extending to Other Domains

Adapting this system for a new domain (e.g. restaurant reviews) requires only two changes:

1. **Data paths** — update `config.DATA_PATH` to point to files that follow the same three-CSV schema (`info`, `desc`, `session`).

2. **Text processor** — subclass `BaseTextProcessor` with domain- or language-specific logic:

```python
from utils.text_processing import BaseTextProcessor, get_text_processor

class SpanishTextProcessor(BaseTextProcessor):
    """Example: Spanish restaurant review processor."""

    def clean_text(self, text): ...
    def normalize(self, text): ...
    def tokenize(self, text): ...
    def remove_stopwords(self, tokens): ...
    def lemmatize(self, tokens): ...
    # process_text() is inherited — no override needed
```

The pipeline in `main.py` and all downstream modules (`topic_modeling`, `recommendation`, `validation`) are fully domain-agnostic.